In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("sci.mplstyle")

In [2]:
def tinh_he_so_backward(kappa, C, rho, N_x, N_time, L, t_max):
    h = L / (N_x - 1) # buoc luoi khong gian
    k = t_max / (N_time - 1) # buoc luoi thoi gian

    alpha = np.sqrt(kappa / (C * rho))

    eta = alpha**2 * k / h**2

    beta1 = eta / (1.0 + 2.0 * eta)
    beta2 = 1.0 / (1.0 + 2.0 * eta)

    return eta, beta1, beta2, h, k

In [3]:
def ham_ghi_file(file, dk_bai_toan): # dung de ghi thong tin quan trong ra file

    file.write(f"# {dk_bai_toan['mo_ta']}\n")
    file.write("# Backward Euler cho phuong trinh truyen nhiet 1D\n")

    file.write("#\n")

    file.write(f"# N_x    = {dk_bai_toan['N_x']}\n")
    file.write(f"# N_time = {dk_bai_toan['N_time']}\n")
    file.write(f"# L      = {dk_bai_toan['L']}\n")
    file.write(f"# t_max  = {dk_bai_toan['t_max']}\n")
    file.write(f"# h      = {dk_bai_toan['h']:.8e}\n")
    file.write(f"# k      = {dk_bai_toan['k']:.8e}\n")
    file.write(f"# eta    = {dk_bai_toan['eta']:.8e}\n")

    file.write("#\n")
    file.write(f"# {'j (t)':>8s} {'i (x)':>8s} {'t':>15s} {'x':>15s} {'u':>15s}\n")

def ham_luu_dulieu(filename, u, dk_bai_toan, skip_buoc_thoigian):
    N_x = dk_bai_toan["N_x"]
    N_time = dk_bai_toan["N_time"]
    h = dk_bai_toan["h"]
    k = dk_bai_toan["k"]

    with open(filename, "w", encoding="utf-8") as file:
        ham_ghi_file(file, dk_bai_toan)

        for j in range(N_time):
            if j % skip_buoc_thoigian != 0 and j != N_time - 1: # khi j khong chia het cho skip_buoc_thoigian thi bo qua data do va lay luon  data cuoi cung
                continue
            t = j * k

            for i in range(N_x):
                x = i * h
                file.write(f"  {j:8d} {i:8d} {t:15.6e} {x:15.6e} {u[i, j]:15.6e}\n")

            file.write("\n")

def ham_luu_hoitu(filename, bang_hoitu): # dung de luu so vong lap hoi tu

    with open(filename, "w", encoding="utf-8") as file:
        file.write("# Ket qua hoi tu cua Backward Euler\n")
        file.write(f"# {'j (t_step)':>15s} {'q (so vong lap)':>15s} {'max_err':>15s}\n")

        for j, q, max_err in bang_hoitu:
            file.write(f"  {j:15d} {q:15d} {max_err:15.6e}\n")

In [4]:
def ham_tinh_backward_euler(u,kappa,C,rho,L,t_max,N_max,tol,filename,mo_ta, skip_buoc_thoigian = 20):

    N_x = u.shape[0]
    N_time = u.shape[1]

    eta, beta1, beta2, h, k = tinh_he_so_backward(
        kappa, C, rho, N_x, N_time, L, t_max
    )

    bang_hoitu = []

    # Vong lap cho thoi gian
    for j in range(1, N_time):

        # Lay nghiem cua buoc truoc lam gia tri doan ban dau
        for i in range(1, N_x - 1):
            u[i, j] = u[i, j - 1]

        # Lap de giai so, voi moi buoc thoi gian thi phai lap de hoi tu tai buoc do
        for q in range(1, N_max + 1):

            u_old = u[:, j].copy()

            for i in range(1, N_x - 1):
                u[i, j] = (
                    beta1 * (u[i + 1, j] + u[i - 1, j])
                    + beta2 * u[i, j - 1]
                )

            max_err = np.max(np.abs(u[:, j] - u_old)) # sai so lon nhat tai moi buoc x (lay lam dai dien cho buoc thoi gian do)

            if max_err < tol:
                bang_hoitu.append((j, q, max_err)) # ghi lai so vong lap cho moi lan hoi tu

                if np.mod(j, 50) == 0: # in ra so vong lap cho moi lan hoi tu
                    print(
                        f"j = {j}, hoi tu sau q = {q} vong lap, "
                        f"max_err = {max_err:.3e}"
                    )

                break
        # Ep 2 dieu kien bien
        u[0, j] = u[0,0]
        u[N_x-1, j] = u[N_x-1, 0]

    dk_bai_toan = {
        "mo_ta": mo_ta,
        "N_x": N_x,
        "N_time": N_time,
        "L": L,
        "t_max": t_max,
        "h": h,
        "k": k,
        "eta": eta,
    }

    ham_luu_dulieu(filename + "_backward_result.txt", u, dk_bai_toan, skip_buoc_thoigian)

    ham_luu_hoitu(filename + "_backward_hoitu.txt", bang_hoitu)

    print("Da tinh xong Backward Euler")
    print(f"eta = {eta:.6e}")
    print(f"beta1 = {beta1:.6e}")
    print(f"beta2 = {beta2:.6e}")
    print(f"Da luu file: {filename + "_backward_result.txt"}")

    return u, dk_bai_toan

In [5]:
# =========================================================
# Bai 1: mot thanh
# =========================================================

def tao_dieu_kien_bien_bai1(N_time, N_x, L, T_ban_dau, T_trai, T_phai):
    """
    Bai 1:
        Thanh dai L
        Ban dau: T(x, 0) = T_ban_dau
        Hai dau: T(0, t) = T_trai, T(L, t) = T_phai

    N_time    : so diem thoi gian
    N_x       : so diem khong gian
    L         : chieu dai thanh
    T_ban_dau : nhiet do ban dau cua thanh
    T_trai    : nhiet do dau trai
    T_phai    : nhiet do dau phai
    """

    x = np.linspace(0.0, L, N_x)
    u = np.zeros((N_x, N_time), dtype=float)

    # Dieu kien dau: T(x, 0) = T_ban_dau
    for i in range(N_x):
        u[i, 0] = T_ban_dau

    # Dieu kien bien trai: T(0, t) = T_trai
    for j in range(N_time):
        u[0, j] = T_trai

    # Dieu kien bien phai: T(L, t) = T_phai
    for j in range(N_time):
        u[N_x - 1, j] = T_phai

    # Tai t = 0, uu tien dieu kien bien tai hai dau
    u[0, 0] = T_trai
    u[N_x - 1, 0] = T_phai

    return u, x

N_time = 6000
N_x = 100

L = 1.0
t_max = 1800.0

T_ban_dau = 100.0
T_trai = 0.0
T_phai = 0.0

kappa = 210.0
C = 900.0
rho = 2700.0

tol  = 1e-6
N_max = 10000

u, x = tao_dieu_kien_bien_bai1(
    N_time=N_time,
    N_x=N_x,
    L=L,
    T_ban_dau=T_ban_dau,
    T_trai=T_trai,
    T_phai=T_phai,
)

# u,kappa,C,rho,L,t_max,N_max,tol,filename,mo_ta, skip_buoc_thoigian = 20

u, info = ham_tinh_backward_euler(
    u=u,
    kappa=kappa,
    C=C,
    rho=rho,
    L=L,
    t_max=t_max,
    N_max=N_max,
    tol = tol,
    filename="truyen_nhiet_1thanh",
    mo_ta="Bai toan truyen nhiet 1D: mot thanh",
    skip_buoc_thoigian = 20,
)

j = 50, hoi tu sau q = 10 vong lap, max_err = 2.105e-07
j = 100, hoi tu sau q = 9 vong lap, max_err = 5.341e-07
j = 150, hoi tu sau q = 9 vong lap, max_err = 3.597e-07
j = 200, hoi tu sau q = 9 vong lap, max_err = 2.706e-07
j = 250, hoi tu sau q = 9 vong lap, max_err = 2.168e-07
j = 300, hoi tu sau q = 8 vong lap, max_err = 8.957e-07
j = 350, hoi tu sau q = 8 vong lap, max_err = 7.685e-07
j = 400, hoi tu sau q = 8 vong lap, max_err = 6.728e-07
j = 450, hoi tu sau q = 8 vong lap, max_err = 5.983e-07
j = 500, hoi tu sau q = 8 vong lap, max_err = 5.389e-07
j = 550, hoi tu sau q = 8 vong lap, max_err = 4.903e-07
j = 600, hoi tu sau q = 8 vong lap, max_err = 4.496e-07
j = 650, hoi tu sau q = 8 vong lap, max_err = 4.151e-07
j = 700, hoi tu sau q = 8 vong lap, max_err = 3.858e-07
j = 750, hoi tu sau q = 8 vong lap, max_err = 3.606e-07
j = 800, hoi tu sau q = 8 vong lap, max_err = 3.385e-07
j = 850, hoi tu sau q = 8 vong lap, max_err = 3.194e-07
j = 900, hoi tu sau q = 8 vong lap, max_err = 3.

In [6]:
# =========================================================
# Bai 2: hai thanh tiep xuc
# =========================================================

# =========================================================
# 3. Tao dieu kien bien bai 2
# =========================================================

def tao_dieu_kien_bien_bai2(
    N_time,
    N_x,
    l_bar,
    T_thanh_trai,
    T_thanh_phai,
    T_bien_trai,
    T_bien_phai,
):
    """
    Bai 2:
        Hai thanh tiep xuc
        Moi thanh dai l_bar
        Tong chieu dai L = 2*l_bar

        Thanh trai ban dau co nhiet do T_thanh_trai
        Thanh phai ban dau co nhiet do T_thanh_phai

        Hai dau ngoai cung giu nhiet do:
            T(0, t) = T_bien_trai
            T(L, t) = T_bien_phai

    N_time       : so diem thoi gian
    N_x          : so diem khong gian
    l_bar        : chieu dai moi thanh
    T_thanh_trai : nhiet do ban dau cua thanh trai
    T_thanh_phai : nhiet do ban dau cua thanh phai
    T_bien_trai  : nhiet do bien trai
    T_bien_phai  : nhiet do bien phai
    """

    L = 2.0 * l_bar
    x = np.linspace(0.0, L, N_x)
    u = np.zeros((N_x, N_time), dtype=float)

    # Dieu kien dau:
    # Neu x <= l_bar thi diem do nam tren thanh trai
    # Neu x >  l_bar thi diem do nam tren thanh phai
    for i in range(N_x):
        if x[i] <= l_bar:
            u[i, 0] = T_thanh_trai
        else:
            u[i, 0] = T_thanh_phai

    # Dieu kien bien trai
    for j in range(N_time):
        u[0, j] = T_bien_trai

    # Dieu kien bien phai
    for j in range(N_time):
        u[N_x - 1, j] = T_bien_phai

    # Tai t = 0, uu tien dieu kien bien tai hai dau
    u[0, 0] = T_bien_trai
    u[N_x - 1, 0] = T_bien_phai

    return u, x, L

N_time = 1500
N_x = 101

l_bar = 0.25
t_max = 1800.0

T_thanh_trai = 100.0
T_thanh_phai = 50.0

T_bien_trai = 0.0
T_bien_phai = 0.0

kappa = 210.0
C = 900.0
rho = 2700.0

tol  = 1e-6
N_max = 10000

u, x, L = tao_dieu_kien_bien_bai2(
    N_time=N_time,
    N_x=N_x,
    l_bar=l_bar,
    T_thanh_trai=T_thanh_trai,
    T_thanh_phai=T_thanh_phai,
    T_bien_trai=T_bien_trai,
    T_bien_phai=T_bien_phai,
)

u, info = ham_tinh_backward_euler(
    u=u,
    kappa=kappa,
    C=C,
    rho=rho,
    L=L,
    t_max=t_max,
    N_max = N_max,
    tol = tol, 
    filename="truyen_nhiet_2thanh",
    mo_ta="Bai toan truyen nhiet 1D: hai thanh tiep xuc",
    skip_buoc_thoigian = 1,
)

j = 50, hoi tu sau q = 55 vong lap, max_err = 8.526e-07
j = 100, hoi tu sau q = 52 vong lap, max_err = 9.544e-07
j = 150, hoi tu sau q = 51 vong lap, max_err = 8.672e-07
j = 200, hoi tu sau q = 50 vong lap, max_err = 8.475e-07
j = 250, hoi tu sau q = 49 vong lap, max_err = 8.477e-07
j = 300, hoi tu sau q = 48 vong lap, max_err = 8.547e-07
j = 350, hoi tu sau q = 47 vong lap, max_err = 8.642e-07
j = 400, hoi tu sau q = 46 vong lap, max_err = 8.744e-07
j = 450, hoi tu sau q = 45 vong lap, max_err = 8.851e-07
j = 500, hoi tu sau q = 44 vong lap, max_err = 8.959e-07
j = 550, hoi tu sau q = 43 vong lap, max_err = 9.068e-07
j = 600, hoi tu sau q = 42 vong lap, max_err = 9.180e-07
j = 650, hoi tu sau q = 41 vong lap, max_err = 9.292e-07
j = 700, hoi tu sau q = 40 vong lap, max_err = 9.406e-07
j = 750, hoi tu sau q = 39 vong lap, max_err = 9.521e-07
j = 800, hoi tu sau q = 38 vong lap, max_err = 9.638e-07
j = 850, hoi tu sau q = 37 vong lap, max_err = 9.757e-07
j = 900, hoi tu sau q = 36 vong 